# Exercises


Exercise 6.1: "Matrix Product"
- Create two large, random square Dask arrays of identical size.
- Time the matrix product between them
- Repeat the above for various combinations of chunk layouts
(Please keep in mind that it is necessary to generate new random arrays each time to ensure that Dask does not "magically" reuse data that has previously been transferred to the client processes.)

In [1]:
import dask.array as da
import time

def time_matrix_product(size, chunk_size):
    # Generating new random arrays each time to avoid cache reuse
    a = da.random.random((size, size), chunks=(chunk_size, chunk_size))
    b = da.random.random((size, size), chunks=(chunk_size, chunk_size))
    
    start = time.time()
    result = da.dot(a, b).compute()
    end = time.time()
    return end - start

size = 4000
for c in [500, 1000, 2000]:
    duration = time_matrix_product(size, c)
    print(f"Size {size}x{size} | Chunks {c}x{c} | Time: {duration:.4f}s")


Size 4000x4000 | Chunks 500x500 | Time: 5.1601s
Size 4000x4000 | Chunks 1000x1000 | Time: 1.6823s
Size 4000x4000 | Chunks 2000x2000 | Time: 1.3779s


Exercise 6.2: "Storing Arrays in HDF5"
- Write a small script that creates a new dask array.
- Fill the Dask array with random values using a function from dask.array.random.
- Store the array to an HDF5 file.
- Repeat the above for different chunk sizes and time how long it takes to generate and store in each case. Does it make any difference?

In [1]:
import dask.array as da
import time
import h5py
import os

def store_hdf5_test(size, chunk_size):
    filename = f"data_{chunk_size}.h5"
    x = da.random.random(size, chunks=chunk_size)
    
    start = time.time()
    da.to_hdf5(filename, '/dataset', x)
    end = time.time()
    
    # Cleanup file
    if os.path.exists(filename): os.remove(filename)
    return end - start

total_elements = 10_000_000
for c in [10_000, 100_000, 1_000_000]:
    duration = store_hdf5_test(total_elements, c)
    print(f"Total Elements: {total_elements} | Chunk Size: {c} | Time: {duration:.4f}s")


Total Elements: 10000000 | Chunk Size: 10000 | Time: 1.1146s
Total Elements: 10000000 | Chunk Size: 100000 | Time: 0.2101s
Total Elements: 10000000 | Chunk Size: 1000000 | Time: 0.1533s


Exercise 6.3: "Pi calculation using Monte Carlo method"
- Implement the calculation of the irrational number Pi using the Monte Carlo method
- This time use the Dask library 
- Measure the execution time and observe the speedup when increasing the number of workers
- How do the results compare to Exercise 3.1?


In [2]:
import dask.array as da
import time
from dask.distributed import Client

# Initialize client to observe workers
client = Client() 

def calculate_pi(n_samples):
    start = time.time()
    
    # Generate points in a 1x1 square
    x = da.random.uniform(0, 1, size=n_samples, chunks=n_samples // 10)
    y = da.random.uniform(0, 1, size=n_samples, chunks=n_samples // 10)
    
    # Count points inside the circle (x^2 + y^2 <= 1)
    inside = (x**2 + y**2 <= 1).sum()
    pi_est = (4 * inside / n_samples).compute()
    
    duration = time.time() - start
    return pi_est, duration

n = 50_000_000
pi_val, dask_time = calculate_pi(n)
print(f"Estimated Pi: {pi_val} | Dask Time: {dask_time:.4f}s")


Estimated Pi: 3.14137816 | Dask Time: 1.8799s
